In [1]:
import pandas as pd

from tradepy.models import *

In [2]:
df = pd.read_csv('../Data/gc1_final.csv')

In [3]:
df["datetime"] = pd.to_datetime(df["datetime"])

# df["ticker"] = df["ticker"].astype("category")
# df["ticker_5min"] = df["ticker_5min"].astype("category")

df = df.sort_values("datetime").reset_index(drop=True)

df = df.drop(columns=["date", "per", "per_5min", 'ticker', 'ticker_5min'])

In [4]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score, mean_absolute_error, mean_absolute_percentage_error, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans, DBSCAN

In [6]:
def linear_regression_model_try(df):
    df = df.copy()

    # 1. Crear target ANTES del dropna
    df["target"] = df["close"].shift(-5) / df["close"] - 1

    # 2. Eliminar filas con NaN (incluye las últimas 5)
    df = df.dropna()

    # 3. Separar X e Y DESPUÉS del dropna
    X = df.drop(columns=["close", "target", "datetime", "close_5min", "close_daily"])
    Y = df["target"]

    # 4. Split temporal (NO aleatorio)
    train_size = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    Y_train, Y_test = Y.iloc[:train_size], Y.iloc[train_size:]

    # 5. Entrenar
    model = LinearRegression()
    model.fit(X_train, Y_train)

    # 6. Evaluar
    Y_pred = model.predict(X_test)
    mse = mean_squared_error(Y_test, Y_pred)
    r2 = r2_score(Y_test, Y_pred)

    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")
    return model


In [7]:
linear_regression_model_try(df)

Mean Squared Error: 1.4853572285264296e-06
R^2 Score: -0.0024217401318051834


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [5]:
import tensorflow as tf


class WindowGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, Y, window_size, batch_size=64):
        self.X = X
        self.Y = Y
        self.window_size = window_size
        self.batch_size = batch_size

    def __len__(self):
        return (len(self.X) - self.window_size) // self.batch_size

    def __getitem__(self, idx):
        X_batch = []
        Y_batch = []

        start = idx * self.batch_size
        end = start + self.batch_size

        for i in range(start, end):
            X_batch.append(self.X[i:i+self.window_size])
            Y_batch.append(self.Y[i+self.window_size])

        return np.array(X_batch), np.array(Y_batch)


def lstm_model_try_2(df, horizon=5, window_size=60):
    df = df.copy()

    # Target futuro
    df["target"] = df["close"].shift(-horizon) / df["close"] - 1
    df = df.dropna().reset_index(drop=True)

    # Features
    feature_cols = [c for c in df.columns if c not in ["datetime", "target"]]
    X = df[feature_cols].values
    Y = df["target"].values

    # Escalado
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Split temporal
    split = int(len(X_scaled) * 0.8)
    X_train, X_test = X_scaled[:split], X_scaled[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    # Generadores
    train_gen = WindowGenerator(X_train, Y_train, window_size)
    test_gen = WindowGenerator(X_test, Y_test, window_size)

    # Modelo LSTM
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(window_size, X.shape[1])),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1)
    ])

    model.compile(optimizer="adam", loss="mse")

    # Entrenamiento
    model.fit(train_gen, epochs=10, validation_data=test_gen)

    # Predicción
    Y_pred = model.predict(test_gen)
    
    # Alinear longitudes
    Y_true = Y_test[window_size:window_size + len(Y_pred)]
    
    # Métricas
    mse = mean_squared_error(Y_true, Y_pred)
    print("MSE:", mse)



In [13]:
lstm_model_try_2(df)

Epoch 1/10


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11334/11334 ━━━━━━━━━━━━━━━━━━━━ 286s 25ms/step - loss: 1.4606e-04 - val_loss: 2.3183e-06
Epoch 2/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 287s 25ms/step - loss: 2.0406e-06 - val_loss: 1.4880e-06
Epoch 3/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 286s 25ms/step - loss: 1.9155e-06 - val_loss: 1.5233e-06
Epoch 4/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 287s 25ms/step - loss: 1.9164e-06 - val_loss: 2.6540e-06
Epoch 5/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 318s 28ms/step - loss: 1.9129e-06 - val_loss: 2.3796e-06
Epoch 6/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 293s 26ms/step - loss: 1.9162e-06 - val_loss: 2.0229e-06
Epoch 7/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 291s 26ms/step - loss: 1.9143e-06 - val_loss: 1.6141e-06
Epoch 8/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 289s 25ms/step - loss: 1.9115e-06 - val_loss: 1.7177e-06
Epoch 9/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 293s 26ms/step - loss: 1.9158e-06 - val_loss: 1.7422e-06
Epoch 10/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 290s 26ms/step - loss: 1.9178e-06 - val_loss: 1.4824e-06


In [6]:
def lstm_model_classification(df, horizon=5, window_size=60, thr=0.0004):
    df = df.copy()

    # ============================
    # 1. Target continuo
    # ============================
    df["target"] = df["close"].shift(-horizon) / df["close"] - 1

    # ============================
    # 2. Target de clasificación (-1,0,1)
    # ============================
    df["target_cls"] = 0
    df.loc[df["target"] > thr, "target_cls"] = 1
    df.loc[df["target"] < -thr, "target_cls"] = -1

    df = df.dropna().reset_index(drop=True)

    # ============================
    # 3. Remapear clases a 0,1,2
    # ============================
    mapping = {-1: 0, 0: 1, 1: 2}
    df["target_cls"] = df["target_cls"].map(mapping)

    # ============================
    # 4. Features
    # ============================
    feature_cols = [c for c in df.columns if c not in ["datetime", "target", "target_cls"]]
    X = df[feature_cols].values
    Y = df["target_cls"].values

    # ============================
    # 5. Escalado
    # ============================
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # ============================
    # 6. Split temporal
    # ============================
    split = int(len(X_scaled) * 0.8)
    X_train, X_test = X_scaled[:split], X_scaled[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    # ============================
    # 7. Generadores
    # ============================
    train_gen = WindowGenerator(X_train, Y_train, window_size)
    test_gen = WindowGenerator(X_test, Y_test, window_size)

    # ============================
    # 8. Modelo LSTM
    # ============================
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(window_size, X.shape[1])),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation="softmax")  # 3 clases
    ])

    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    # ============================
    # 9. Entrenamiento
    # ============================
    model.fit(train_gen, epochs=10, validation_data=test_gen)

    # ============================
    # 10. Predicción
    # ============================
    Y_pred_proba = model.predict(test_gen)
    Y_pred = np.argmax(Y_pred_proba, axis=1)

    # ============================
    # 11. Remapear de vuelta a (-1,0,1)
    # ============================
    inv_mapping = {0: -1, 1: 0, 2: 1}
    Y_pred = np.array([inv_mapping[c] for c in Y_pred])

    # ============================
    # 12. Alinear longitudes
    # ============================
    Y_true = Y_test[window_size:window_size + len(Y_pred)]
    Y_true = np.array([inv_mapping[c] for c in Y_true])

    # ============================
    # 13. Métricas
    # ============================
    print("Accuracy:", accuracy_score(Y_true, Y_pred))
    print(classification_report(Y_true, Y_pred))

    return model


In [21]:
lstm_model_classification(df)

Epoch 1/10


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11334/11334 ━━━━━━━━━━━━━━━━━━━━ 348s 30ms/step - accuracy: 0.4529 - loss: 1.0476 - val_accuracy: 0.4413 - val_loss: 1.0515
Epoch 2/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 280s 25ms/step - accuracy: 0.4594 - loss: 1.0409 - val_accuracy: 0.4409 - val_loss: 1.0507
Epoch 3/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 279s 25ms/step - accuracy: 0.4603 - loss: 1.0399 - val_accuracy: 0.4437 - val_loss: 1.0482
Epoch 4/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 278s 25ms/step - accuracy: 0.4611 - loss: 1.0390 - val_accuracy: 0.4432 - val_loss: 1.0479
Epoch 5/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 279s 25ms/step - accuracy: 0.4623 - loss: 1.0377 - val_accuracy: 0.4420 - val_loss: 1.0482
Epoch 6/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 279s 25ms/step - accuracy: 0.4627 - loss: 1.0363 - val_accuracy: 0.4458 - val_loss: 1.0480
Epoch 7/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 279s 25ms/step - accuracy: 0.4636 - loss: 1.0351 - val_accuracy: 0.4456 - val_loss: 1.0432
Epoch 8/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 279s 25ms/step - accur

<Sequential name=sequential_6, built=True>

In [ ]:
def lstm_model_classification_2(df, horizon=5, window_size=60, thr=0.0004, epochs=5):
    df = df.copy()

    # Target continuo
    df["target"] = df["close"].shift(-horizon) / df["close"] - 1

    # Target de clasificación (-1,0,1)
    df["target_cls"] = 0
    df.loc[df["target"] > thr, "target_cls"] = 1
    df.loc[df["target"] < -thr, "target_cls"] = -1

    df = df.dropna().reset_index(drop=True)

    # Remapear clases a 0,1,2
    mapping = {-1: 0, 0: 1, 1: 2}
    inv_mapping = {0: -1, 1: 0, 2: 1}
    df["target_cls"] = df["target_cls"].map(mapping)

    # Features
    feature_cols = [c for c in df.columns if c not in ["datetime", "target", "target_cls"]]
    X = df[feature_cols].values
    Y = df["target_cls"].values

    # Escalado
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Split temporal
    split = int(len(X_scaled) * 0.8)
    X_train, X_test = X_scaled[:split], X_scaled[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    # Generadores
    train_gen = WindowGenerator(X_train, Y_train, window_size)
    test_gen = WindowGenerator(X_test, Y_test, window_size)

    # Modelo
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(window_size, X.shape[1])),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation="softmax")
    ])

    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_gen, epochs=epochs, validation_data=test_gen)

    return model, test_gen, Y_test, inv_mapping


In [35]:
import os

def grid_search_lstm_classification(
    df,
    window_sizes=[12, 24, 36, 48, 60],
    thresholds=[0.0002, 0.0004, 0.0006, 0.0010],
    horizon=5,
    epochs=5,
    results_path="grid_results.csv"
):
    if not os.path.exists(results_path):
        pd.DataFrame(columns=["window", "thr", "accuracy", "f1_macro"]).to_csv(results_path, index=False)
    results = []

    for w in window_sizes:
        for thr in thresholds:
            print(f"\n=== Probando window={w}, thr={thr} ===")

            model, test_gen, Y_test, inv_mapping = lstm_model_classification_2(
                df=df,
                horizon=horizon,
                window_size=w,
                thr=thr,
                epochs=epochs
            )

            # Predicciones
            Y_pred_proba = model.predict(test_gen)
            Y_pred = np.argmax(Y_pred_proba, axis=1)
            Y_pred = np.array([inv_mapping[c] for c in Y_pred])

            # Alinear longitudes
            Y_true = Y_test[w:w + len(Y_pred)]
            Y_true = np.array([inv_mapping[c] for c in Y_true])

            # Métricas
            acc = accuracy_score(Y_true, Y_pred)
            f1 = f1_score(Y_true, Y_pred, average="macro")

            results.append({
                "window": w,
                "thr": thr,
                "accuracy": acc,
                "f1_macro": f1
            })

            print(f"Accuracy={acc:.4f}, F1={f1:.4f}")
            
            new_row = pd.DataFrame([{
                "window": w,
                "thr": thr,
                "accuracy": acc,
                "f1_macro": f1
            }])

            new_row.to_csv(results_path, mode="a", header=False, index=False)

    return sorted(results, key=lambda x: x["f1_macro"], reverse=True)


In [36]:
results = grid_search_lstm_classification(df)


=== Probando window=12, thr=0.0002 ===
Epoch 1/5


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11335/11335 ━━━━━━━━━━━━━━━━━━━━ 115s 10ms/step - accuracy: 0.4021 - loss: 1.0526 - val_accuracy: 0.4092 - val_loss: 1.0419
Epoch 2/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 108s 9ms/step - accuracy: 0.4067 - loss: 1.0488 - val_accuracy: 0.4023 - val_loss: 1.0428
Epoch 3/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 111s 10ms/step - accuracy: 0.4103 - loss: 1.0479 - val_accuracy: 0.4050 - val_loss: 1.0409
Epoch 4/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 107s 9ms/step - accuracy: 0.4106 - loss: 1.0476 - val_accuracy: 0.4080 - val_loss: 1.0446
Epoch 5/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 105s 9ms/step - accuracy: 0.4109 - loss: 1.0471 - val_accuracy: 0.3977 - val_loss: 1.0469
2833/2833 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step
Accuracy=0.3977, F1=0.2022

=== Probando window=12, thr=0.0004 ===
Epoch 1/5


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11335/11335 ━━━━━━━━━━━━━━━━━━━━ 106s 9ms/step - accuracy: 0.4535 - loss: 1.0467 - val_accuracy: 0.4411 - val_loss: 1.0509
Epoch 2/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 104s 9ms/step - accuracy: 0.4596 - loss: 1.0408 - val_accuracy: 0.4442 - val_loss: 1.0476
Epoch 3/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 110s 10ms/step - accuracy: 0.4604 - loss: 1.0397 - val_accuracy: 0.4444 - val_loss: 1.0497
Epoch 4/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 91s 8ms/step - accuracy: 0.4610 - loss: 1.0391 - val_accuracy: 0.4431 - val_loss: 1.0502
Epoch 5/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 77s 7ms/step - accuracy: 0.4606 - loss: 1.0383 - val_accuracy: 0.4445 - val_loss: 1.0456
2833/2833 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step
Accuracy=0.4445, F1=0.3868

=== Probando window=12, thr=0.0006 ===
Epoch 1/5


...

11334/11334 ━━━━━━━━━━━━━━━━━━━━ 282s 25ms/step - accuracy: 0.4531 - loss: 1.0466 - val_accuracy: 0.4431 - val_loss: 1.0481
Epoch 2/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 283s 25ms/step - accuracy: 0.4586 - loss: 1.0410 - val_accuracy: 0.4414 - val_loss: 1.0549
Epoch 3/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 281s 25ms/step - accuracy: 0.4599 - loss: 1.0398 - val_accuracy: 0.4457 - val_loss: 1.0458
Epoch 4/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 282s 25ms/step - accuracy: 0.4602 - loss: 1.0390 - val_accuracy: 0.4423 - val_loss: 1.0498
Epoch 5/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 324s 29ms/step - accuracy: 0.4611 - loss: 1.0380 - val_accuracy: 0.4419 - val_loss: 1.0506
2833/2833 ━━━━━━━━━━━━━━━━━━━━ 29s 10ms/step
Accuracy=0.4419, F1=0.3207

=== Probando window=60, thr=0.0006 ===
Epoch 1/5


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11334/11334 ━━━━━━━━━━━━━━━━━━━━ 274s 24ms/step - accuracy: 0.5689 - loss: 0.9444 - val_accuracy: 0.5531 - val_loss: 0.9540
Epoch 2/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 266s 23ms/step - accuracy: 0.5709 - loss: 0.9364 - val_accuracy: 0.5559 - val_loss: 0.9617
Epoch 3/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 265s 23ms/step - accuracy: 0.5711 - loss: 0.9352 - val_accuracy: 0.5558 - val_loss: 0.9544
Epoch 4/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 265s 23ms/step - accuracy: 0.5721 - loss: 0.9335 - val_accuracy: 0.5564 - val_loss: 0.9480
Epoch 5/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 264s 23ms/step - accuracy: 0.5719 - loss: 0.9313 - val_accuracy: 0.5544 - val_loss: 0.9482
2833/2833 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step
Accuracy=0.5544, F1=0.3704

=== Probando window=60, thr=0.001 ===
Epoch 1/5


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11334/11334 ━━━━━━━━━━━━━━━━━━━━ 270s 24ms/step - accuracy: 0.7445 - loss: 0.6919 - val_accuracy: 0.7432 - val_loss: 0.6855
Epoch 2/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 263s 23ms/step - accuracy: 0.7450 - loss: 0.6820 - val_accuracy: 0.7450 - val_loss: 0.6854
Epoch 3/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 262s 23ms/step - accuracy: 0.7450 - loss: 0.6796 - val_accuracy: 0.7438 - val_loss: 0.6888
Epoch 4/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 262s 23ms/step - accuracy: 0.7451 - loss: 0.6779 - val_accuracy: 0.7438 - val_loss: 0.6826
Epoch 5/5
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 262s 23ms/step - accuracy: 0.7454 - loss: 0.6759 - val_accuracy: 0.7441 - val_loss: 0.6789
2833/2833 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step
Accuracy=0.7441, F1=0.3237


In [37]:
def backtest_single(df, window, thr, horizon=5, fees=4.0, specs=None):
    """
    specs = {
        "tick_size": 0.10,
        "tick_value": 10.00,
        "contract_unit": 100,
        "leverage_typical": 10
    }
    """

    model, test_gen, Y_test, inv_mapping = lstm_model_classification_2(
        df=df,
        horizon=horizon,
        window_size=window,
        thr=thr,
        epochs=5
    )

    # Predicciones
    Y_pred_proba = model.predict(test_gen)
    Y_pred = np.argmax(Y_pred_proba, axis=1)
    Y_pred = np.array([inv_mapping[c] for c in Y_pred])

    # Alinear
    Y_true = Y_test[window:window + len(Y_pred)]
    Y_true = np.array([inv_mapping[c] for c in Y_true])

    # Señales
    signals = Y_pred  # -1, 0, 1

    # Precios
    close = df["close"].values
    close_test = close[len(close) - len(Y_test):]  # parte test
    entry_prices = close_test[window:window + len(signals)]
    exit_prices = close_test[window + horizon:window + horizon + len(signals)]

    # PnL
    pnl_usd = []
    pnl_pct = []

    for s, entry, exit_ in zip(signals, entry_prices, exit_prices):
        if s == 0:
            pnl_usd.append(0)
            pnl_pct.append(0)
            continue

        price_move = (exit_ - entry) * s
        ticks = price_move / specs["tick_size"]
        pnl = ticks * specs["tick_value"] - fees
        pnl_usd.append(pnl)

        pct = (pnl / (entry * specs["contract_unit"])) * specs["leverage_typical"]
        pnl_pct.append(pct)

    pnl_usd = np.array(pnl_usd)
    pnl_pct = np.array(pnl_pct)

    # Métricas
    total_return = pnl_pct.sum()
    sharpe = pnl_pct.mean() / (pnl_pct.std() + 1e-9)
    winrate = (pnl_usd > 0).mean()
    trades = (signals != 0).sum()

    return {
        "window": window,
        "thr": thr,
        "return_pct": total_return,
        "sharpe": sharpe,
        "winrate": winrate,
        "trades": trades
    }


In [38]:

def backtest_all(df, csv_path, specs, top=None):
    results = pd.read_csv(csv_path)

    # Ordenar por F1 macro
    results = results.sort_values("f1_macro", ascending=False)

    if top is not None:
        results = results.head(top)

    backtest_results = []

    for _, row in results.iterrows():
        print(f"Backtesting window={row.window}, thr={row.thr}")

        bt = backtest_single(
            df=df,
            window=int(row.window),
            thr=float(row.thr),
            specs=specs
        )

        backtest_results.append(bt)

    return pd.DataFrame(backtest_results)


In [8]:
model, test_gen, Y_test, inv_mapping = lstm_model_classification_2(df=df, horizon=5, window_size=36, thr=0.0004, epochs=5)

c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 191s 17ms/step - accuracy: 0.4531 - loss: 1.0471 - val_accuracy: 0.4340 - val_loss: 1.0589
Epoch 2/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 190s 17ms/step - accuracy: 0.4594 - loss: 1.0411 - val_accuracy: 0.4438 - val_loss: 1.0490
Epoch 3/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 186s 16ms/step - accuracy: 0.4602 - loss: 1.0400 - val_accuracy: 0.4427 - val_loss: 1.0481
Epoch 4/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 189s 17ms/step - accuracy: 0.4607 - loss: 1.0387 - val_accuracy: 0.4426 - val_loss: 1.0474
Epoch 5/5
11335/11335 ━━━━━━━━━━━━━━━━━━━━ 187s 16ms/step - accuracy: 0.4613 - loss: 1.0380 - val_accuracy: 0.4418 - val_loss: 1.0490


In [9]:
model.save("model_w36_thr0004.keras")

In [10]:
def backtest_single_loaded(df, model_path, window, thr, horizon=5, fees=4.0, specs=None):
    # Cargar modelo ya entrenado
    model = tf.keras.models.load_model(model_path)

    # Regenerar test_gen y Y_test (pero sin entrenar)
    _, test_gen, Y_test, inv_mapping = lstm_model_classification_2(
        df=df,
        horizon=horizon,
        window_size=window,
        thr=thr,
        epochs=0   # IMPORTANTE: no entrenar
    )

    # Predicciones
    Y_pred_proba = model.predict(test_gen)
    Y_pred = np.argmax(Y_pred_proba, axis=1)
    Y_pred = np.array([inv_mapping[c] for c in Y_pred])

    # Alinear
    Y_true = Y_test[window:window + len(Y_pred)]
    Y_true = np.array([inv_mapping[c] for c in Y_true])

    # Señales
    signals = Y_pred  # -1, 0, 1

    # Precios
    close = df["close"].values
    close_test = close[len(close) - len(Y_test):]
    entry_prices = close_test[window:window + len(signals)]
    exit_prices = close_test[window + horizon:window + horizon + len(signals)]

    # PnL
    pnl_usd = []
    pnl_pct = []

    for s, entry, exit_ in zip(signals, entry_prices, exit_prices):
        if s == 0:
            pnl_usd.append(0)
            pnl_pct.append(0)
            continue

        price_move = (exit_ - entry) * s
        ticks = price_move / specs["tick_size"]
        pnl = ticks * specs["tick_value"] - fees
        pnl_usd.append(pnl)

        pct = (pnl / (entry * specs["contract_unit"])) * specs["leverage_typical"]
        pnl_pct.append(pct)

    pnl_usd = np.array(pnl_usd)
    pnl_pct = np.array(pnl_pct)

    # Métricas
    total_return = pnl_pct.sum()
    sharpe = pnl_pct.mean() / (pnl_pct.std() + 1e-9)
    winrate = (pnl_usd > 0).mean()
    trades = (signals != 0).sum()

    return {
        "window": window,
        "thr": thr,
        "return_pct": total_return,
        "sharpe": sharpe,
        "winrate": winrate,
        "trades": trades
    }


In [11]:
specs = {
    "tick_size": 0.10,
    "tick_value": 10.00,
    "contract_unit": 100,
    "leverage_typical": 10
}

bt = backtest_single_loaded(
    df=df,
    model_path="model_w36_thr0004.keras",
    window=36,
    thr=0.0004,
    specs=specs
)

bt


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2833/2833 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step


{'window': 36,
 'thr': 0.0004,
 'return_pct': np.float64(-20.397565543442767),
 'sharpe': np.float64(-0.013425440484749886),
 'winrate': np.float64(0.21593165372396753),
 'trades': np.int64(81064)}

In [19]:
def backtest_single_filtered(df, model_path, window, thr, horizon=5, fees=4.0, specs=None,
                             min_persistence=3, min_prob=0.6):

    model = tf.keras.models.load_model(model_path)

    _, test_gen, Y_test, inv_mapping = lstm_model_classification_2(
        df=df,
        horizon=horizon,
        window_size=window,
        thr=thr,
        epochs=0
    )

    # Probabilidades
    Y_pred_proba = model.predict(test_gen)
    Y_pred_raw = np.argmax(Y_pred_proba, axis=1)
    Y_pred_raw = np.array([inv_mapping[c] for c in Y_pred_raw])

    # --- FILTRO DE PROBABILIDAD ---
    Y_pred = []
    for proba, cls in zip(Y_pred_proba, Y_pred_raw):
        if np.max(proba) < min_prob:
            Y_pred.append(0)  # señal débil → no operar
        else:
            Y_pred.append(cls)

    Y_pred = np.array(Y_pred)

    # --- FILTRO DE PERSISTENCIA ---
    filtered = []
    last = None
    count = 0

    for s in Y_pred:
        if s == last:
            count += 1
        else:
            last = s
            count = 1

        if count >= min_persistence:
            filtered.append(s)
        else:
            filtered.append(0)

    filtered = np.array(filtered)

    # Precios
    close = df["close"].values
    close_test = close[len(close) - len(Y_test):]

    entry_prices = close_test[window:window + len(filtered)]
    exit_prices  = close_test[window + horizon:window + horizon + len(filtered)]

    pnl_usd = []
    pnl_pct = []

    last_signal = 0
    real_trades = 0

    for s, entry, exit_ in zip(filtered, entry_prices, exit_prices):

        if s != last_signal:
            real_trades += 1
            last_signal = s
        else:
            pnl_usd.append(0)
            pnl_pct.append(0)
            continue

        if s == 0:
            pnl_usd.append(0)
            pnl_pct.append(0)
            continue

        price_move = (exit_ - entry) * s
        ticks = price_move / specs["tick_size"]
        pnl = ticks * specs["tick_value"] - fees
        pnl_usd.append(pnl)

        pct = (pnl / (entry * specs["contract_unit"])) * specs["leverage_typical"]
        pnl_pct.append(pct)

    pnl_usd = np.array(pnl_usd)
    pnl_pct = np.array(pnl_pct)

    return {
        "window": window,
        "thr": thr,
        "return_pct": pnl_pct.sum(),
        "sharpe": pnl_pct.mean() / (pnl_pct.std() + 1e-9),
        "winrate": (pnl_usd > 0).mean(),
        "trades": real_trades
    }


In [21]:
specs = {
    "tick_size": 0.10,
    "tick_value": 10.00,
    "contract_unit": 100,
    "leverage_typical": 10
}

bt1 = backtest_single_filtered(
    df=df,
    model_path="model_w36_thr0004.keras",
    window=36,
    thr=0.0004,
    specs=specs,
    min_prob=0.5
)

bt1


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2833/2833 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step


{'window': 36,
 'thr': 0.0004,
 'return_pct': np.int64(0),
 'sharpe': np.float64(0.0),
 'winrate': np.float64(0.0),
 'trades': 0}

In [ ]:
specs = {
    "tick_size": 0.10,
    "tick_value": 10.00,
    "contract_unit": 100,
    "leverage_typical": 10
}

bt_results = backtest_all(df, "grid_results.csv", specs, top=5)
bt_results


#

In [22]:
def rolling_walkforward_splits(df, train_bars, test_bars, step_bars):
    """
    Devuelve una lista de (start_train, end_train, start_test, end_test)
    índices sobre df, para rolling + walk-forward.
    """
    n = len(df)
    splits = []

    start_train = 0
    end_train = train_bars

    while True:
        start_test = end_train
        end_test = end_train + test_bars

        if end_test > n:
            break

        splits.append((start_train, end_train, start_test, end_test))

        # avanzar ventana
        start_train += step_bars
        end_train += step_bars

    return splits


In [23]:
def lstm_train_on_segment(df_segment, horizon, window_size, thr, epochs):
    df_seg = df_segment.copy()

    df_seg["target"] = df_seg["close"].shift(-horizon) / df_seg["close"] - 1
    df_seg["target_cls"] = 0
    df_seg.loc[df_seg["target"] > thr, "target_cls"] = 1
    df_seg.loc[df_seg["target"] < -thr, "target_cls"] = -1

    df_seg = df_seg.dropna().reset_index(drop=True)

    mapping = {-1: 0, 0: 1, 1: 2}
    inv_mapping = {0: -1, 1: 0, 2: 1}
    df_seg["target_cls"] = df_seg["target_cls"].map(mapping)

    feature_cols = [c for c in df_seg.columns if c not in ["datetime", "target", "target_cls"]]
    X = df_seg[feature_cols].values
    Y = df_seg["target_cls"].values

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    train_gen = WindowGenerator(X_scaled, Y, window_size)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(window_size, X.shape[1])),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation="softmax")
    ])

    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    model.fit(train_gen, epochs=epochs, verbose=0)

    return model, scaler, inv_mapping, feature_cols


In [24]:
def backtest_segment(df_test, model, scaler, inv_mapping, feature_cols,
                     window_size, horizon, specs, fees=4.0):
    df_t = df_test.copy()

    df_t["target"] = df_t["close"].shift(-horizon) / df_t["close"] - 1
    df_t = df_t.dropna().reset_index(drop=True)

    X = df_t[feature_cols].values
    X_scaled = scaler.transform(X)

    # construir ventanas a mano para test
    X_seq = []
    for i in range(len(X_scaled) - window_size - horizon):
        X_seq.append(X_scaled[i:i+window_size])

    X_seq = np.array(X_seq)

    Y_pred_proba = model.predict(X_seq, verbose=0)
    Y_pred_cls = np.argmax(Y_pred_proba, axis=1)
    Y_pred = np.array([inv_mapping[c] for c in Y_pred_cls])  # -1,0,1

    close = df_t["close"].values
    entry_prices = close[window_size:window_size + len(Y_pred)]
    exit_prices  = close[window_size + horizon:window_size + horizon + len(Y_pred)]

    pnl_usd = []
    pnl_pct = []
    last_signal = 0
    trades = 0

    for s, entry, exit_ in zip(Y_pred, entry_prices, exit_prices):
        if s == 0 or s == last_signal:
            pnl_usd.append(0)
            pnl_pct.append(0)
            last_signal = s
            continue

        trades += 1
        last_signal = s

        price_move = (exit_ - entry) * s
        ticks = price_move / specs["tick_size"]
        pnl = ticks * specs["tick_value"] - fees
        pnl_usd.append(pnl)

        pct = (pnl / (entry * specs["contract_unit"])) * specs["leverage_typical"]
        pnl_pct.append(pct)

    pnl_usd = np.array(pnl_usd)
    pnl_pct = np.array(pnl_pct)

    if trades == 0:
        return {
            "return_pct": 0.0,
            "sharpe": 0.0,
            "winrate": 0.0,
            "trades": 0
        }

    return {
        "return_pct": pnl_pct.sum(),
        "sharpe": pnl_pct.mean() / (pnl_pct.std() + 1e-9),
        "winrate": (pnl_usd > 0).mean(),
        "trades": trades
    }


In [25]:
def rolling_walkforward_backtest(df, train_bars, test_bars, step_bars,
                                 horizon, window_size, thr, epochs, specs):
    splits = rolling_walkforward_splits(df, train_bars, test_bars, step_bars)

    results = []

    for i, (st_tr, en_tr, st_te, en_te) in enumerate(splits):
        print(f"Bloque {i+1}: train[{st_tr}:{en_tr}], test[{st_te}:{en_te}]")

        df_train = df.iloc[st_tr:en_tr]
        df_test  = df.iloc[st_te:en_te]

        model, scaler, inv_mapping, feature_cols = lstm_train_on_segment(
            df_segment=df_train,
            horizon=horizon,
            window_size=window_size,
            thr=thr,
            epochs=epochs
        )

        bt = backtest_segment(
            df_test=df_test,
            model=model,
            scaler=scaler,
            inv_mapping=inv_mapping,
            feature_cols=feature_cols,
            window_size=window_size,
            horizon=horizon,
            specs=specs
        )

        bt["block"] = i + 1
        bt["train_start"] = st_tr
        bt["train_end"] = en_tr
        bt["test_start"] = st_te
        bt["test_end"] = en_te

        results.append(bt)

        tf.keras.backend.clear_session()

    return pd.DataFrame(results)


In [ ]:
specs = {
    "tick_size": 0.10,
    "tick_value": 10.00,
    "contract_unit": 100,
    "leverage_typical": 10
}

res = rolling_walkforward_backtest(
    df=df,
    train_bars=20000,   # ajusta según tu timeframe
    test_bars=2000,
    step_bars=2000,
    horizon=5,
    window_size=36,
    thr=0.0004,
    epochs=5,
    specs=specs
)

res
res[["block", "return_pct", "sharpe", "winrate", "trades"]]
